# Deep Past Challenge: Old Assyrian to English Translation

## Overview

This notebook provides an exploratory data analysis of the Deep Past Challenge dataset, which contains transliterations of Old Assyrian cuneiform tablets with English translations.

The challenge: Akkadian is a low-resource, morphologically complex language where a single word can encode what takes multiple words in English. Standard architectures built for modern, data-rich languages fail here.

### Dataset Description
- **train_augmented.csv**: Augmented training data combining original + external sources (~28,000 samples)
- **test.csv**: Test set for predictions (~4 sample sentences in this notebook)
- **Supplemental data**: Published texts, sentence-level translations, and scholarly publications
   
### External Resources Used for Augmentation
The augmented dataset combines the original Deep Past Challenge data with:
- **Akkademia Dataset** (Gutherz et al. 2023): ~27,000 samples from ORACC RINAP projects
- Source: https://github.com/gaigutherz/Akkademia
- Sentence-level translations from Sentences_Oare_FirstWord_LinNum.csv

### Test Data Structure
The test data contains sentence-level translations with text_id, line_start, and line_end fields
indicating the boundaries of each sentence within the original tablet.

### Data Preprocessing Notes
Per the DATA.md formatting suggestions:
- Training data uses Ḫ ḫ characters while test data uses H h - substitution needed
- Various transliteration markers need handling (! ? / : < > [ ] { })
- Line numbers use str type with values like 1, 1', or 1''


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load the Data

In [ ]:
# Load the training and test datasets
train_df = pd.read_csv('train_augmented.csv')
test_df = pd.read_csv('test.csv')

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"\nTraining data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")

print("\n--- Training Data Columns ---")
print(train_df.columns.tolist())

print("\n--- Test Data Columns ---")
print(test_df.columns.tolist())

## 2. Examine the Training Data

In [ ]:
# Display first few rows
train_df.head(3)

In [ ]:
# Check for missing values
print("Missing values in training data:")
print(train_df.isnull().sum())

print("\n--- Sample Transliteration ---")
print(train_df['transliteration'].iloc[0][:200])

print("\n--- Sample Translation ---")
print(train_df['translation'].iloc[0][:500])

## 3. Text Length Analysis

In [ ]:
# Calculate text lengths
train_df['transliteration_len'] = train_df['transliteration'].str.len()
train_df['translation_len'] = train_df['translation'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Transliteration length distribution
axes[0].hist(train_df['transliteration_len'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Character Length')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Transliteration Lengths')
axes[0].axvline(train_df['transliteration_len'].mean(), color='red', linestyle='--', label=f'Mean: {train_df["transliteration_len"].mean():.0f}')
axes[0].legend()

# Translation length distribution
axes[1].hist(train_df['translation_len'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Character Length')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Translation Lengths')
axes[1].axvline(train_df['translation_len'].mean(), color='red', linestyle='--', label=f'Mean: {train_df["translation_len"].mean():.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('text_length_distributions.png', dpi=150)
plt.show()

print("\n--- Text Length Statistics ---")
print(train_df[['transliteration_len', 'translation_len']].describe())

## 4. Length Ratio Analysis

In [ ]:
# Calculate length ratio (translation / transliteration)
train_df['length_ratio'] = train_df['translation_len'] / train_df['transliteration_len']

plt.figure(figsize=(10, 5))
plt.hist(train_df['length_ratio'], bins=50, edgecolor='black', alpha=0.7, color='purple')
plt.xlabel('Length Ratio (Translation / Transliteration)')
plt.ylabel('Frequency')
plt.title('Translation to Transliteration Length Ratio')
plt.axvline(train_df['length_ratio'].mean(), color='red', linestyle='--', 
            label=f'Mean: {train_df["length_ratio"].mean():.2f}')
plt.legend()
plt.savefig('length_ratio_distribution.png', dpi=150)
plt.show()

print(f"\nMean length ratio: {train_df['length_ratio'].mean():.2f}")
print(f"This indicates translations are on average {train_df['length_ratio'].mean():.1f}x longer than transliterations")

## 5. Test Data Analysis

In [ ]:
print("Test data:")
print(test_df)

print("\n--- Sample Test Transliterations ---")
for i, row in test_df.iterrows():
    print(f"\nID {row['id']} (text_id: {row['text_id']}):")
    print(f"  Lines: {row['line_start']}-{row['line_end']}")
    print(f"  Text: {row['transliteration'][:100]}...")

## 7. Explore Supplemental Data

In [ ]:
# Load supplemental datasets
published_texts = pd.read_csv('published_texts.csv')
sentences = pd.read_csv('Sentences_Oare_FirstWord_LinNum.csv')

print("=" * 60)
print("SUPPLEMENTAL DATA OVERVIEW")
print("=" * 60)

print(f"\nPublished texts: {len(published_texts)} entries")
print(f"Sentences with translations: {len(sentences)} entries")

print("\n--- Published Texts Columns ---")
print(published_texts.columns.tolist())

print("\n--- Sentences Columns ---")
print(sentences.columns.tolist())

## 8. Key Findings Summary

In [ ]:
print("=" * 60)
print("KEY FINDINGS FROM EXPLORATORY DATA ANALYSIS")
print("=" * 60)

print(f"""
1. TRAINING DATA:
   - Total samples: {len(train_df)}
   - Average transliteration length: {train_df['transliteration_len'].mean():.0f} characters
   - Average translation length: {train_df['translation_len'].mean():.0f} characters
   - Mean length ratio: {train_df['length_ratio'].mean():.2f}x

2. DATA CHARACTERISTICS:
   - Training data has document-level translations (per DATA.md)
   - Test data has sentence-level translations with text_id, line_start, line_end
   - Test contains ~4000 sentences from ~400 unique documents

3. TEST DATA:
   - Test samples: {len(test_df)}
   - Note: These are sample sentences from one document (text_id: {test_df['text_id'].iloc[0]})
4. SUPPLEMENTAL DATA:
   - Published texts (without translations): {len(published_texts)}
   - Sentences with translations: {len(sentences)}

5. CHALLENGES:
   - Low-resource language (~1,500 training examples)
   - Morphologically complex language
   - Translations vary significantly in length
   - Training data uses Ḫ ḫ but test uses H h - preprocessing needed
""")